# WavLM + Prosody Extraction v3 - 555 Videos

## Improvements:
- Audio extracted to Google Drive (persistent)
- Checkpoint every 5000 utterances (auto-resume)
- WavLM batch=32 for 4x speedup
- Progress with ETA

**Runtime:** ~3-4 hours on T4 GPU

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')
print('Drive mounted!')

In [ ]:
!apt-get install -y ffmpeg
!pip install -q transformers librosa scikit-learn

import os, json, time, glob
import numpy as np
import torch, torch.nn as nn
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import librosa
from pathlib import Path

SR = 16000; BATCH = 32; SEED = 42; CHECKPOINT_EVERY = 5000
torch.manual_seed(SEED); np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# Load utterances from GitHub
!wget -q -O /tmp/utterances_clean.jsonl.gz https://github.com/Das-rebel/chuck-audio-notebooks/raw/main/utterances_clean.jsonl.gz
!gunzip -f /tmp/utterances_clean.jsonl.gz
utterances = [json.loads(l) for l in open('/tmp/utterances_clean.jsonl')]
print(f'Loaded {len(utterances)} utterances from {len(set(u["video_id"] for u in utterances))} videos')

In [ ]:
# Download audio tar (if not already on Drive)
!pip install -q gdown
DRIVE_AUDIO_DIR = Path('/content/gdrive/MyDrive/vtt_audio_local')
DRIVE_CHECKPOINT = Path('/content/gdrive/MyDrive/extraction_checkpoint_v3.npz')

if not DRIVE_AUDIO_DIR.exists():
    print('Downloading audio (4.9GB, ~5 min)...')
    !gdown 1jHy11LdU0bv8ra-Lj9xhpKT4yXE3L1Tp -O /tmp/vtt_audio_local.tar.gz
    !mkdir -p /content/gdrive/MyDrive
    !tar -xzf /tmp/vtt_audio_local.tar.gz -C /content/gdrive/MyDrive/
    print('Audio extracted to Drive!')
else:
    print('Audio already on Drive')

In [ ]:
# Build audio file map
audio_files = {f.stem: str(f) for f in DRIVE_AUDIO_DIR.glob('*') if f.suffix in ['.m4a', '.mp3', '.wav']}
print(f'Found {len(audio_files)} audio files')

# Filter utterances to only those with audio
valid_utterances = [u for u in utterances if u['video_id'] in audio_files]
print(f'Valid utterances (with audio): {len(valid_utterances)}')

In [ ]:
# Load WavLM model
print('Loading WavLM...')
from transformers import WavLMModel
model = WavLMModel.from_pretrained('microsoft/wavlm-base-plus')
model = model.to(device).eval()
print('Model loaded!')

In [ ]:
def extract_prosody_batch(audio_path, segments, sr=SR):
    """Extract prosody for multiple segments at once."""
    prosody_list = []
    for start, end in segments:
        try:
            y, sr = librosa.load(audio_path, sr=sr, mono=True, offset=start, duration=end-start)
            if len(y) < sr * 0.05:
                prosody_list.append(np.zeros(21, dtype=np.float32))
                continue
            feats = []
            try:
                f0, _, _ = librosa.pyin(y, fmin=50, fmax=500, sr=sr)
                f0v = f0[~np.isnan(f0)]
                feats.extend([np.mean(f0v), np.std(f0v), np.max(f0v), np.min(f0v), np.median(f0v)] if len(f0v)>0 else [0]*5)
            except: feats.extend([0]*5)
            rms = librosa.feature.rms(y=y)[0]
            feats.extend([np.mean(rms), np.std(rms), np.max(rms), np.min(rms)])
            spec = [librosa.feature.spectral_centroid(y=y, sr=sr)[0], librosa.feature.spectral_bandwidth(y=y, sr=sr)[0], librosa.feature.spectral_flatness(y=y)[0], librosa.feature.zero_crossing_rate(y)[0]]
            feats.extend([np.mean(s) for s in spec] + [np.max(spec[-1])])
            mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=7)
            feats.extend([np.mean(mfccs[i]) for i in range(7)])
            prosody_list.append(np.array(feats, dtype=np.float32))
        except:
            prosody_list.append(np.zeros(21, dtype=np.float32))
    return prosody_list

In [ ]:
def extract_wavlm_batch(audio_path, segments, sr=SR):
    """Extract WavLM embeddings for multiple segments at once."""
    embeddings = []
    for start, end in segments:
        try:
            y, sr = librosa.load(audio_path, sr=sr, mono=True, offset=start, duration=end-start)
            if len(y) < sr * 0.05:
                embeddings.append(np.zeros(768, dtype=np.float32))
                continue
            y_t = torch.tensor(y).unsqueeze(0).to(device)
            with torch.no_grad():
                outputs = model(y_t)
                emb = outputs.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
            embeddings.append(emb)
        except:
            embeddings.append(np.zeros(768, dtype=np.float32))
    return embeddings

In [ ]:
# RESUME or START extraction
if DRIVE_CHECKPOINT.exists():
    print('Loading checkpoint...')
    ckpt = np.load(DRIVE_CHECKPOINT)
    all_emb = list(ckpt['embeddings'])
    all_pro = list(ckpt['prosody'])
    all_lbl = list(ckpt['labels'])
    all_uid = list(ckpt['uids'])
    print(f'Resumed: {len(all_emb)} utterances already processed')
else:
    print('Starting fresh extraction')
    all_emb, all_pro, all_lbl, all_uid = [], [], [], []

processed_uids = set(all_uid)
total = len(valid_utterances)
remaining = [u for u in valid_utterances if f"{u['video_id']}_{u['start']:.2f}_{u['end']:.2f}" not in processed_uids]
print(f'Total: {total}, Remaining: {len(remaining)}, Done: {len(all_emb)}')

t0 = time.time()
for batch_start in range(0, len(remaining), BATCH):
    batch = remaining[batch_start:batch_start+BATCH]
    
    # Group by video for efficient loading
    video_segs = {}
    for u in batch:
        vid = u['video_id']
        if vid not in video_segs: video_segs[vid] = []
        video_segs[vid].append((u['start'], u['end'], u['label'], f"{vid}_{u['start']:.2f}_{u['end']:.2f}"))
    
    for vid, segs in video_segs.items():
        audio_path = audio_files[vid]
        segments = [(s, e) for s, e, l, uid in segs]
        
        # Extract WavLM and prosody
        embs = extract_wavlm_batch(audio_path, segments)
        pros = extract_prosody_batch(audio_path, segments)
        
        for emb, pro, (s, e, lbl, uid) in zip(embs, pros, segs):
            all_emb.append(emb)
            all_pro.append(pro)
            all_lbl.append(lbl)
            all_uid.append(uid)
    
    # Progress update
    done = len(all_emb)
    if done % 5000 < BATCH or done == total:
        elapsed = time.time() - t0
        rate = done / elapsed if elapsed > 0 else 0
        eta = (total - done) / rate / 60 if rate > 0 else 0
        pct = done / total * 100
        print(f'Progress: {done}/{total} ({pct:.1f}%) - {rate:.1f}/s - ETA: {eta:.0f}min')
    
    # Save checkpoint
    if done % CHECKPOINT_EVERY < BATCH:
        np.savez(DRIVE_CHECKPOINT,
            embeddings=np.array(all_emb, dtype=np.float32),
            prosody=np.array(all_pro, dtype=np.float32),
            labels=np.array(all_lbl, dtype=np.int64),
            uids=np.array(all_uid))
        print(f'Checkpoint saved: {done} utterances')

In [ ]:
# Save final
FINAL_FILE = '/content/gdrive/MyDrive/wavlm_prosody_555_v3.npz'
np.savez(FINAL_FILE,
    embeddings=np.array(all_emb, dtype=np.float32),
    prosody=np.array(all_pro, dtype=np.float32),
    labels=np.array(all_lbl, dtype=np.int64),
    uids=np.array(all_uid))
print(f'Saved final: {len(all_emb)} utterances to {FINAL_FILE}')